# Week 2 Assignment · Build RAG From Parts — and Prove It Works
### Build Custom AI — SarasAI · *ungraded practice*

The live session built a RAG pipeline in front of you. Here you build every part yourself —
**11 tasks in four parts** — and, more importantly, you *measure* each part before trusting it:

| Part | Tasks | What you build |
|---|---|---|
| 1 · Corpus & index | 1–3 | a heterogeneous corpus (reviews + specs) · embeddings · FAISS index |
| 2 · Retrieval, measured | 4–6 | `retrieve()` · **recall@k on a gold set** · the BGE-prefix A/B |
| 3 · Grounded generation | 7–9 | `generate()` · grounded prompt with citations · the refusal-rule stress test |
| 4 · Evaluation | 10–11 | with/without-retrieval table · an LLM **faithfulness judge** |

Every task has `...` placeholders; most are followed by a **self-check cell**. Stuck ≥15 min?
Read *that one task* in the solution notebook, close it, write your own. ⏱ ~60–90 min on a T4.

---
## Setup (given)

The corpus: **24 customer reviews + 4 internal documents** (3 spec sheets, 1 policy) about
three fictional products. Fictional = you always know the ground truth, and anything specific
the bare model claims later is guaranteed hallucination.

In [ ]:
!pip install -q "transformers>=4.50" sentence-transformers faiss-cpu accelerate

In [ ]:
import torch

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}   dtype: {DTYPE}")

In [ ]:
REVIEWS = [
    ("r01", "The KT-2000 kettle looks great but the lid hinge snapped after two weeks."),
    ("r02", "KT-2000 boils fast. However the lid cracked near the hinge within a month."),
    ("r03", "Love my KT-2000, no issues after six months of daily use."),
    ("r04", "Kettle handle gets warm but never hot. KT-2000 is solid."),
    ("r05", "The KT-2000 lid mechanism feels flimsy; mine broke at the hinge too."),
    ("r06", "KT-2000 descaling is easy, just vinegar and a rinse."),
    ("r07", "TS-400 toaster: the lever sticks halfway. Annoying every morning."),
    ("r08", "My TS-400 lever jams unless I press it twice. Bought June 2023."),
    ("r09", "TS-400 toasts evenly, best toaster I have owned."),
    ("r10", "The TS-400 crumb tray is easy to clean. No complaints."),
    ("r11", "Lever on the TS-400 started sticking after about a month of use."),
    ("r12", "TS-400 bagel mode is perfect. The dial feels premium."),
    ("r13", "BL-900 blender crushes ice fine, but the jar lid seal drips."),
    ("r14", "My BL-900 leaks from the lid seal when blending soups."),
    ("r15", "BL-900 motor is powerful, smoothie in 20 seconds."),
    ("r16", "The BL-900 jar seal let go mid-blend. Kitchen ceiling casualty."),
    ("r17", "BL-900 is loud but effective. Would buy again."),
    ("r18", "Blender base wobbles slightly on my counter, BL-900."),
    ("r19", "Customer service replaced my KT-2000 lid for free, great support."),
    ("r20", "TS-400 heating element failed after two years, fair lifespan."),
    ("r21", "The BL-900 pulse button is mushy but works."),
    ("r22", "KT-2000 auto-shutoff works exactly as advertised."),
    ("r23", "Gifted a TS-400, recipient loves it apart from the sticky lever."),
    ("r24", "BL-900 blades dulled after a year of daily smoothies."),
]

SPECS = [
    ("spec_kettle", "KT-2000 electric kettle. Capacity 1.7 L. Body: borosilicate glass. "
                    "Lid assembly: polypropylene, part PP-114. Warranty: 24 months."),
    ("spec_toaster", "TS-400 two-slot toaster. Lever mechanism part SP-77C revised in 2023-06 "
                     "to fix reported sticking. Warranty: 12 months."),
    ("spec_blender", "BL-900 blender. Motor 900 W. Jar 1.5 L with silicone lid seal, part LS-22. "
                     "Jar is dishwasher safe up to 60 degrees. Warranty: 12 months on motor."),
    ("policy_returns", "Return policy for all products (KT-2000, TS-400, BL-900): defective units "
                       "returnable within 30 days for full refund; after 30 days warranty claims "
                       "are repair or replacement. Water damage and misuse are excluded."),
]
print(len(REVIEWS), "reviews +", len(SPECS), "internal docs")

---
## Part 1 · Corpus & index

### ✏️ Task 1 — one corpus, uniform shape

> 💡 **Hint:** two list comprehensions and a +. Keep the ids — they become citation labels.

In [ ]:
# ╔══════════════════════ TASK 1 ══════════════════════╗
# one corpus, uniform shape
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# Merge reviews and specs into one list of dicts: {"id": ..., "text": ..., "kind": "review"|"doc"}
# Retrieval doesn't care where text came from — but citations and debugging do.
corpus = ...

assert len(corpus) == 28
print(corpus[0], "\n", corpus[-1])

### ✏️ Task 2 — embed the corpus — and verify the vectors are unit-length

> 💡 **Hint:** normalize_embeddings=True; np.linalg.norm(matrix, axis=1) gives one norm per row.

In [ ]:
# ╔══════════════════════ TASK 2 ══════════════════════╗
# embed the corpus — and verify the vectors are unit-length
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

# (a) encode every corpus text, L2-normalized, as float32
doc_vecs = ...

# (b) verify: with normalized vectors, inner product == cosine similarity.
#     Check that every vector's L2 norm is ~1.0
norms = ...
print(f"shape: {doc_vecs.shape}   norms: min {norms.min():.4f}  max {norms.max():.4f}")

In [ ]:
# ── self-check ──
assert doc_vecs.shape == (28, 384), "expected 28 vectors of dim 384 (bge-small)"
assert abs(float(norms.max()) - 1.0) < 1e-3, "vectors must be unit-length — did you normalize?"
print("✅ 28 unit vectors — inner product now IS cosine similarity")

### ✏️ Task 3 — the FAISS index

> 💡 **Hint:** faiss.IndexFlatIP(dim) then index.add(matrix). Two lines.

In [ ]:
# ╔══════════════════════ TASK 3 ══════════════════════╗
# the FAISS index
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
import faiss

# build an inner-product index of the right dimension and add the vectors
index = ...
...

print("indexed:", index.ntotal, "vectors of dim", index.d)

---
## Part 2 · Retrieval — never trust it unmeasured

A retriever you haven't measured is a rumor. You'll build it, then score it on a **gold set**
before any LLM gets involved.

### ✏️ Task 4 — `retrieve(question, k)` with the BGE query prefix

> 💡 **Hint:** index.search(query_matrix, k) returns (scores, indices), both shaped (1, k).

In [ ]:
# ╔══════════════════════ TASK 4 ══════════════════════╗
# `retrieve(question, k)` with the BGE query prefix
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# BGE embeds QUERIES better with this instruction prefix (documents stay unprefixed):
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(question, k=3, prefix=BGE_PREFIX):
    """Top-k corpus entries as [{"id", "text", "kind", "score"}, ...]."""
    q = ...                                  # encode prefix + question (normalized, float32)
    scores, idxs = ...                       # search the index
    return [...]                             # build result dicts from corpus[i] + score

for hit in retrieve("kettle lid broke"):
    print(f"{hit['score']:.3f}  {hit['id']:13}  {hit['text'][:55]}")

In [ ]:
# ── self-check: semantic, not keyword ──
ids = {h["id"] for h in retrieve("blender leaks liquid everywhere", k=3)}
assert ids & {"r13", "r14", "r16"}, f"expected a BL-900 seal review in top-3, got {ids}"
print("✅ retrieval finds meaning ('leaks liquid' ≈ 'seal drips') — not just keywords")

### ✏️ Task 5 — recall@k on a gold set

> 💡 **Hint:** set(item['sources']) <= retrieved_ids tests 'all required sources present'.

In [ ]:
# ╔══════════════════════ TASK 5 ══════════════════════╗
# recall@k on a gold set
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# 6 gold questions. `sources` = ids that MUST be retrieved for the answer to be possible.
GOLD = [
    {"q": "What is the warranty period for the KT-2000 kettle?",   "sources": ["spec_kettle"]},
    {"q": "Which part fixes the TS-400 lever sticking?",           "sources": ["spec_toaster"]},
    {"q": "Is the BL-900 jar dishwasher safe?",                    "sources": ["spec_blender"]},
    {"q": "Can I return a defective unit after two weeks?",        "sources": ["policy_returns"]},
    # for review questions, ANY of the listed sources counts (r01/r02/r05 are all lid reviews)
    {"q": "What do customers say breaks on the KT-2000?",          "sources": ["r01", "r02", "r05"], "any": True},
    {"q": "Do customers report the BL-900 leaking?",               "sources": ["r13", "r14", "r16"], "any": True},
]

def recall_at_k(gold, k, prefix=BGE_PREFIX):
    """Fraction of gold questions satisfied in the top-k.
    Default: ALL listed sources must be retrieved. Rows with "any": True need just one."""
    hits = 0
    for item in gold:
        retrieved_ids = ...                  # set of ids from retrieve(item["q"], k, prefix)
        hits += ...                          # all sources present — or any, if item.get("any")
    return hits / len(gold)

for k in [1, 3, 5]:
    print(f"recall@{k}: {recall_at_k(GOLD, k):.0%}")

In [ ]:
# ── self-check + how to READ the numbers ──
r5 = recall_at_k(GOLD, 5)
assert r5 >= 0.5, f"recall@5 = {r5:.0%} — something is off in retrieve() or the index"
print(f"✅ recall@5 = {r5:.0%}")
print("Diagnostic reading (from the session): recall@5 high but recall@1 low -> embeddings fine,")
print("consider a reranker. recall@5 low -> fix chunking/embeddings BEFORE touching any prompt.")

### ✏️ Task 6 — does the BGE prefix actually matter? Measure it

> 💡 **Hint:** your retrieve() already takes prefix as a parameter — that was the point.

In [ ]:
# ╔══════════════════════ TASK 6 ══════════════════════╗
# does the BGE prefix actually matter? Measure it
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# Same gold set, k=3: score retrieval WITH the prefix and WITHOUT (prefix="").
with_prefix    = ...
without_prefix = ...

print(f"recall@3 with prefix:    {with_prefix:.0%}")
print(f"recall@3 without prefix: {without_prefix:.0%}")

On 6 questions the gap may be zero or one hit — tiny samples are noisy (Week 1's lesson).
The habit is what matters: **config choices get measured, not assumed.** On the graded
increment's ≥20-question gold set, differences like this become visible.

---
## Part 3 · Grounded generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(GEN_ID)
gen_lm = AutoModelForCausalLM.from_pretrained(GEN_ID, torch_dtype=DTYPE, device_map="auto")
print(f"generator loaded: {gen_lm.num_parameters()/1e9:.2f}B params")

### ✏️ Task 7 — the `generate()` chat helper

> 💡 **Hint:** out[0][inputs['input_ids'].shape[1]:] slices off the prompt tokens before decoding.

In [ ]:
# ╔══════════════════════ TASK 7 ══════════════════════╗
# the `generate()` chat helper
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
def generate(user_text, max_new_tokens=200):
    """One chat turn with the generator LLM, greedy decoding."""
    # (a) apply_chat_template on a single user message
    #     (add_generation_prompt=True, return_dict=True, return_tensors="pt")
    inputs = ...
    # (b) generate (do_sample=False, pad_token_id=gen_tok.eos_token_id)
    out = ...
    # (c) decode ONLY the newly generated tokens (slice off the prompt), skip special tokens
    return ...

print(generate("Say READY if you can hear me.", max_new_tokens=5))

### ✏️ Task 8 — grounded prompt + `rag_answer()`

> 💡 **Hint:** context first, rules second, question last — the order the session taught.

In [ ]:
# ╔══════════════════════ TASK 8 ══════════════════════╗
# grounded prompt + `rag_answer()`
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
def rag_answer(question, k=3):
    """Retrieve, then answer FROM the retrieved text only, citing [ids]."""
    hits = retrieve(question, k)
    context = "\n".join(f"[{h['id']}] {h['text']}" for h in hits)

    # Three blocks, in this order: CONTEXT, RULES, QUESTION.
    # Rules must demand: answer only from context · cite [ids] after each claim ·
    # if the context doesn't cover it, reply exactly "I don't have enough information".
    prompt = ...

    return generate(prompt), hits

answer, hits = rag_answer("What breaks on the KT-2000 kettle?")
print(answer)

In [ ]:
# ── self-check: citations must reference retrieved ids ──
# lenient on FORM ([r01], (r01), "source r01" all count) — strict on SUBSTANCE (a real id)
import re
cited = set(re.findall(r"\b(r\d{2}|spec_\w+|policy_\w+)\b", answer))
retrieved = {h["id"] for h in hits}
if cited & retrieved:
    print(f"✅ grounded: cites {cited & retrieved}")
else:
    print(f"⚠️ no retrieved id found in the answer (retrieved: {retrieved}).")
    print("   Tighten your RULES line: demand bracket citations like [r01] after each claim, and re-run.")

### ✏️ Task 9 — stress-test the refusal rule

> 💡 **Hint:** if the rate is low, tighten your RULES block in Task 8 and re-run — that loop IS prompt debugging.

In [ ]:
# ╔══════════════════════ TASK 9 ══════════════════════╗
# stress-test the refusal rule
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# Three questions the corpus CANNOT answer. A production RAG must refuse all three.
ADVERSARIAL = [
    "What colors does the KT-2000 come in?",
    "How much does the BL-900 cost?",
    "Who is the CEO of the company that makes the TS-400?",
]

def refuses(answer):
    """True if the answer is a refusal rather than an invention."""
    return ...                    # look for your refusal phrase (case-insensitive, be lenient)

refusal_rate = ...                # fraction of ADVERSARIAL answered with a refusal
print(f"refusal rate on unanswerable questions: {refusal_rate:.0%}")

In [ ]:
# ── see what it actually said ──
for q in ADVERSARIAL:
    a, _ = rag_answer(q)
    print(f"{'✅ refused ' if refuses(a) else '❌ INVENTED'}  {q}\n     -> {a[:90]}\n")
print("A RAG system that can't say 'I don't know' is a liability, not an assistant.")

---
## Part 4 · The evaluation that sells it

### ✏️ Task 10 — with vs. without retrieval, side by side

> 💡 **Hint:** the 'without' arm is literally generate(q) — that IS the point.

In [ ]:
# ╔══════════════════════ TASK 10 ══════════════════════╗
# with vs. without retrieval, side by side
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
QUESTIONS = [
    "What is the most common complaint about the TS-400 toaster?",
    "Does the BL-900 blender have a known leaking problem?",
    "What is the warranty period for the KT-2000 kettle?",
]

for q in QUESTIONS:
    bare = ...          # the generator alone — no context at all
    grounded, _ = ...   # your rag_answer
    print(f"Q: {q}")
    print(f"  WITHOUT: {bare[:140]}")
    print(f"  WITH   : {grounded[:140]}")
    print("-" * 80)

### ✏️ Task 11 — a faithfulness judge

> 💡 **Hint:** one-word verdicts parse reliably; 'YES' in verdict.upper() is enough.

In [ ]:
# ╔══════════════════════ TASK 11 ══════════════════════╗
# a faithfulness judge
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
def judge_faithful(question, answer, context):
    """1 if every claim in `answer` is supported by `context`, else 0 — via the LLM itself."""
    # Build a judge prompt: show context + answer, ask "is every claim supported by the
    # context? Reply exactly one word: YES or NO." Parse leniently.
    prompt = ...
    verdict = generate(prompt, max_new_tokens=5)
    return ...

faithful = 0
for item in GOLD:
    ans, hits = rag_answer(item["q"])
    ctx = "\n".join(h["text"] for h in hits)
    faithful += judge_faithful(item["q"], ans, ctx)
print(f"faithfulness: {faithful}/{len(GOLD)}")

In [ ]:
# ── honest caveat (given — read it) ──
print("""Judge caveats from the session — they apply to YOUR number above:
 1. This judge shares weights with the generator -> self-agreement bias. Real evals use a
    STRONGER, DIFFERENT judge model (or the `ragas` library).
 2. A 1.5B judge is a smoke detector, not an auditor.
 3. Faithfulness != correctness: a faithful answer to badly-retrieved context is still wrong.
    That's why recall@k (Task 5) exists as a separate, retrieval-only number.""")

---
## Wrap-up · What you practiced

| Task | Skill | Where it goes next |
|---|---|---|
| 1–3 | corpus shaping, embeddings, FAISS | graded increment 2's index |
| 4–6 | retrieve(), **recall@k**, config A/B | the retrieval half of your eval |
| 7–9 | generate(), grounding, refusal stress test | the generation half |
| 10–11 | with/without table, faithfulness judge | your stakeholder demo + RAGAS onramp |

**Reflection (2 min, edit into a cell):** your recall@1 vs recall@5 — which diagnostic did the
gap suggest (reranker vs. fix embeddings)? And on which adversarial question did the refusal
rule fail first?

*Ungraded — nothing to submit. Graded increment 2 = the full capstone corpus, ≥20-question
frozen gold set, recall@k table, faithfulness + answer-relevance via `ragas`.*